# Ajuste de hiperparámetros

## Qué responde este notebook

[`4.2_modelos_recepciones_yardas.ipynb`](4.2_modelos_recepciones_yardas.ipynb) y
[`4.3_modelos_touchdowns.ipynb`](4.3_modelos_touchdowns.ipynb) eligieron XGBoost para los tres
resultados con una grilla pequeña (profundidad, número de árboles y tasa de aprendizaje). Aquí no se
vuelven a comparar familias; se prueba si el mismo XGBoost mejora con una búsqueda compacta de
hiperparámetros, en la que cada combinación encuentra su número de árboles con *early stopping*,
usando la misma métrica que decide (RMSE o deviance de Poisson). En touchdowns se busca además la
potencia de Tweedie, comparada contra el objetivo Poisson. Las variables son las 25 compartidas de
[`3.3_seleccion_estable.ipynb`](3.3_seleccion_estable.ipynb) (sección 8 del
[glosario](../../../docs/data/GLOSARIO_VARIABLES.md)): el conjunto compartido por los tres
resultados es parte de esa decisión, así que aquí no se prueban conjuntos propios por resultado.

Un cambio se adopta solo si mejora la métrica principal en validación **y** la mejora es real: el
intervalo bootstrap por semanas de la diferencia, con el nivel ajustado por las combinaciones
probadas (ver [`4.1_preparacion_y_metricas.ipynb`](4.1_preparacion_y_metricas.ipynb)), no incluye
cero. La
búsqueda es compacta para acotar el cómputo; `experimentos.ESPACIO_HIPERPARAMETROS_XGBOOST` define un
espacio más amplio (regularización y submuestreo) por si se quiere explorar más.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

from itertools import product

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

wr = features.construir_tabla_modelado(range(2016, 2026))
features_compartidas = features.columnas_por_modelo("arbol")
train_mask, val_mask, test_mask = experimentos.dividir_temporal(
    wr, range(2016, 2022), [2022, 2023], [2024, 2025]
)
TARGETS = list(modeling.TARGETS)

def evaluar(target, columnas, params):
    modelo = XGBRegressor(**params, random_state=0, n_jobs=-1)
    return experimentos.evaluar_modelo(modelo, wr[columnas], wr[target], train_mask, val_mask, test_mask)

def comparar(target, pred_nueva, pred_actual):
    return experimentos.diferencia_bootstrap(
        wr[val_mask], wr.loc[val_mask, target], pred_nueva[val_mask], pred_actual[val_mask],
        modeling.METRICA_PRINCIPAL[target])

## 1. El punto de partida

El XGBoost elegido en 4.2 y 4.3: la misma grilla de 12 combinaciones, elegida por la métrica
principal en validación. En touchdowns, con objetivo `count:poisson`, el que quedó primero.

In [2]:
GRILLA = list(product((3, 4, 6), (200, 400), (0.03, 0.08)))
OBJETIVO = {"receptions": {}, "receiving_yards": {}, "receiving_tds": {"objective": "count:poisson"}}

punto_partida = {}
for target in TARGETS:
    criterio = modeling.METRICA_PRINCIPAL[target]
    mejor = None
    for d, n, lr in GRILLA:
        params = {**OBJETIVO[target], "max_depth": d, "n_estimators": n, "learning_rate": lr}
        salida = evaluar(target, features_compartidas, params)
        if mejor is None or salida["val"][criterio] < mejor["salida"]["val"][criterio]:
            mejor = {"params": params, "salida": salida}
    punto_partida[target] = mejor
    print(f"{target}: {mejor['params']} | {criterio} de validacion = {mejor['salida']['val'][criterio]:.4f}")

receptions: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03} | rmse de validacion = 1.9315


receiving_yards: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03} | rmse de validacion = 29.5002


receiving_tds: {'objective': 'count:poisson', 'max_depth': 3, 'n_estimators': 400, 'learning_rate': 0.03} | deviance_poisson de validacion = 0.6128


## 2. Ajuste compacto de hiperparámetros

- **Recepciones y yardas**: profundidad 3, 4 o 5 y tasa 0.03 o 0.05 (6 combinaciones), con early
  stopping sobre RMSE de validación.
- **Touchdowns**: potencia de Tweedie 1.2 a 1.8 y profundidad 2 a 4 (15 combinaciones), más el
  objetivo Poisson con profundidad 2 a 4 (3 combinaciones), con tasa 0.03 y early stopping sobre la
  log-verosimilitud de Poisson, que equivale a la deviance salvo una constante.

In [3]:
ajustados = {}
for target in TARGETS:
    criterio = modeling.METRICA_PRINCIPAL[target]
    busquedas = [experimentos.buscar_hiperparametros(wr, target, features_compartidas, train_mask, val_mask)]
    if target == "receiving_tds":
        busquedas.append(experimentos.buscar_hiperparametros(
            wr, target, features_compartidas, train_mask, val_mask,
            espacio={"max_depth": [2, 3, 4]}, params_fijos={"objective": "count:poisson"}))
    modelo, config, _ = min(busquedas, key=lambda b: b[2][f"val_{criterio}"].iloc[0])
    pred = pd.Series(modeling.clip_no_negativo(modelo.predict(wr[features_compartidas])), index=wr.index)
    ajustados[target] = {"config": config, "pred": pred,
                         "intentos": pd.concat([b[2] for b in busquedas], ignore_index=True)}
    print(f"{target}: mejor combinacion {{" + ", ".join(f"{k}={config[k]}" for k in
          ("objective", "tweedie_variance_power", "max_depth", "learning_rate", "n_estimators") if k in config) + "}")

columnas_intentos = ["objective", "tweedie_variance_power", "max_depth", "learning_rate", "n_estimators_optimo",
                     "val_deviance_poisson", "val_d2_oos", "val_sesgo", "val_pendiente_calibracion"]
ajustados["receiving_tds"]["intentos"].reindex(columns=columnas_intentos).sort_values("val_deviance_poisson").head(8).round(4)

receptions: mejor combinacion {max_depth=3, learning_rate=0.05, n_estimators=100}


receiving_yards: mejor combinacion {max_depth=3, learning_rate=0.05, n_estimators=130}


receiving_tds: mejor combinacion {objective=reg:tweedie, tweedie_variance_power=1.5, max_depth=4, learning_rate=0.03, n_estimators=81}


,objective,tweedie_variance_power,max_depth,learning_rate,n_estimators_optimo,val_deviance_poisson,val_d2_oos,val_sesgo,val_pendiente_calibracion
0,reg:tweedie,1.50,4,0.03,81,0.6118,0.1292,0.0120,1.0884
1,reg:tweedie,1.20,2,0.03,359,0.6120,0.1290,0.0166,0.9957
15,count:poisson,NaN,2,0.03,528,0.6121,0.1288,0.0190,1.0419
2,reg:tweedie,1.35,2,0.03,273,0.6123,0.1284,0.0147,1.0292
3,reg:tweedie,1.50,2,0.03,229,0.6124,0.1283,0.0136,1.0418
4,reg:tweedie,1.50,3,0.03,120,0.6125,0.1282,0.0125,1.0652
5,reg:tweedie,1.20,3,0.03,209,0.6125,0.1281,0.0151,1.0064
16,count:poisson,NaN,3,0.03,460,0.6126,0.1281,0.0189,1.0049


In [4]:
n_configuraciones = sum(len(ajustados[t]["intentos"]) for t in TARGETS)
nivel = 1 - 0.05 / n_configuraciones
print(f"configuraciones probadas en los tres resultados: {n_configuraciones}; nivel ajustado: {nivel:.4f}")

filas = {}
for target in TARGETS:
    criterio = modeling.METRICA_PRINCIPAL[target]
    actual = punto_partida[target]["salida"]
    medida_nueva = modeling.metricas_target(target, wr.loc[val_mask, target], ajustados[target]["pred"][val_mask],
                                            media_entrenamiento=actual["media_entrenamiento"])
    c = experimentos.diferencia_bootstrap(wr[val_mask], wr.loc[val_mask, target], ajustados[target]["pred"][val_mask],
                                          actual["pred"][val_mask], criterio, n_remuestreos=20000, nivel=nivel)
    c95 = comparar(target, ajustados[target]["pred"], actual["pred"])
    ajustados[target]["adoptar"] = (not c["empate"]) and c["diferencia"] < 0
    filas[target] = {"criterio": criterio, "punto de partida": actual["val"][criterio], "ajustado": medida_nueva[criterio],
                     "diferencia": c["diferencia"], "IC 95% inferior": c95["ic_inferior"], "IC 95% superior": c95["ic_superior"],
                     "IC ajustado inferior": c["ic_inferior"], "IC ajustado superior": c["ic_superior"],
                     "se adopta": ajustados[target]["adoptar"]}
pd.DataFrame(filas).T

configuraciones probadas en los tres resultados: 30; nivel ajustado: 0.9983


,criterio,punto de partida,ajustado,diferencia,IC 95% inferior,IC 95% superior,IC ajustado inferior,IC ajustado superior,se adopta
receptions,rmse,1.931488,1.929246,-0.002242,-0.004376,-0.000134,-0.005795,0.001252,False
receiving_yards,rmse,29.50024,29.498043,-0.002197,-0.026093,0.02204,-0.042472,0.034442,False
receiving_tds,deviance_poisson,0.612773,0.611808,-0.000965,-0.003915,0.001917,-0.005795,0.004067,False


**Lectura.** El ajuste compacto mejora la métrica principal en milésimas en los tres resultados
(recepciones −0.0022 de RMSE, yardas −0.0022, touchdowns −0.0010 de deviance). Cada resultado se
queda con la mejor de varias combinaciones (30 en total), así que, con la regla de 4.1, el intervalo
se calcula al 99.83%. Con ese nivel las tres diferencias incluyen cero y no se adopta ninguna. En
recepciones, el intervalo de 95% sin ajustar sí excluye el cero (de −0.0044 a −0.0001): es justo el
caso para el que existe la regla, una mejora del tamaño de lo que puede aparecer por azar al quedarse
con la mejor de varias combinaciones.

En touchdowns la mejor combinación fue Tweedie con potencia 1.5 y profundidad 4, pero las ocho
mejores quedan entre 0.6118 y 0.6126, prácticamente igual que el punto de partida con objetivo
Poisson (0.6128): con estas variables, la grilla original ya queda en lo mejor que encuentra la
búsqueda.

## 4. Configuración elegida por resultado

La que queda después del ajuste, con su revisión de requisitos en validación. La
prueba se reporta aquí por primera vez para estas configuraciones; la evaluación año por año está en
[`5.1_estabilidad_temporal.ipynb`](5.1_estabilidad_temporal.ipynb).

In [5]:
elegida = {}
for target in TARGETS:
    if ajustados[target]["adoptar"]:
        cfg = {k: v for k, v in ajustados[target]["config"].items() if k not in ("random_state", "n_jobs")}
        origen = "ajuste compacto"
    else:
        cfg = punto_partida[target]["params"]
        origen = "punto de partida"
    salida = evaluar(target, features_compartidas, cfg)
    elegida[target] = {"params": cfg, "origen": origen, "salida": salida}
    cumple, detalle = experimentos.cumple_requisitos(target, salida["val"])
    print(f"{target}: {origen} -> {cfg}")
    print(f"   requisitos en validacion: {'cumple' if cumple else 'NO cumple'}")
    for chequeo, ok in detalle.items():
        print(f"      {'ok ' if ok else 'NO '} {chequeo}")

columnas = ["rmse", "r2_oos", "deviance_poisson", "d2_oos", "sesgo", "pendiente_calibracion", "mae"]
pd.concat({parte: pd.DataFrame({t: {k: elegida[t]["salida"][parte].get(k) for k in columnas} for t in TARGETS}).T
           for parte in ("val", "test")}, axis=1).astype(float).round(4)

receptions: punto de partida -> {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03}
   requisitos en validacion: cumple
      ok  R2 fuera de muestra > 0 (validacion)
      ok  sesgo adicional al de la media de entrenamiento <= 5% de la media (validacion)
      ok  pendiente de calibracion entre 0.9 y 1.1 (validacion)


receiving_yards: punto de partida -> {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.03}
   requisitos en validacion: cumple
      ok  R2 fuera de muestra > 0 (validacion)
      ok  sesgo adicional al de la media de entrenamiento <= 5% de la media (validacion)
      ok  pendiente de calibracion entre 0.9 y 1.1 (validacion)


receiving_tds: punto de partida -> {'objective': 'count:poisson', 'max_depth': 3, 'n_estimators': 400, 'learning_rate': 0.03}
   requisitos en validacion: cumple
      ok  R2 fuera de muestra > 0 (validacion)
      ok  D2 fuera de muestra > 0 (validacion)
      ok  sesgo adicional al de la media de entrenamiento <= 5% de la media (validacion)
      ok  pendiente de calibracion entre 0.9 y 1.1 (validacion)


val                                                                             test                                           \
                    rmse  r2_oos deviance_poisson  d2_oos   sesgo pendiente_calibracion      mae     rmse  r2_oos deviance_poisson  d2_oos   sesgo   
receptions        1.9315  0.4364           1.4498  0.4294  0.0919                1.0474   1.4502   1.8355  0.4527           1.3980  0.4493  0.0577   
receiving_yards  29.5002  0.3655              NaN     NaN  1.4722                1.0667  21.9857  27.9349  0.3744              NaN     NaN  1.2251   
receiving_tds     0.4329  0.0903           0.6128  0.1278  0.0196                1.0199   0.3032   0.4454  0.0914           0.6221  0.1399 -0.0044   

                                                
                pendiente_calibracion      mae  
receptions                     1.0212   1.3697  
receiving_yards                1.0159  20.5522  
receiving_tds                  1.0736   0.3020

**Lectura.** Quedan las configuraciones de partida: XGBoost de profundidad 3, 200 árboles y tasa
0.03 para recepciones y yardas, y XGBoost con objetivo Poisson, profundidad 3, 400 árboles y tasa
0.03 para touchdowns. Las tres cumplen los requisitos en validación. En prueba quedan al nivel de
validación o un poco arriba: R² fuera de muestra de 0.4527 en recepciones (0.4364 en validación),
0.3744 en yardas (0.3655) y 0.0914 en touchdowns (0.0903), con D² de 0.1399 en touchdowns
(0.1278) y pendientes de calibración entre 1.02 y 1.07. En touchdowns el sesgo de prueba es pequeño
(−0.0044).

## Cierre

Un ajuste más fino de hiperparámetros no mejora de forma real a XGBoost con la grilla original. Quedan esas configuraciones, que cumplen los requisitos. Los
notebooks 4.5 a 4.9 prueban alternativas de modelado contra ellas, y la etapa 5 revisa su estabilidad
año por año, las compara contra el flujo que existía antes de este proyecto y explica sus predicciones.

Anterior: [`4.3_modelos_touchdowns.ipynb`](4.3_modelos_touchdowns.ipynb) · Siguiente:
[`4.5_regresion_cuantiles.ipynb`](4.5_regresion_cuantiles.ipynb)